# Cloud RAG Demo

This notebook demonstrates how the RAG pipeline answers queries using the vector database.

## 🔍 What this demonstrates

- End-to-end RAG pipeline
- Topic-aware retrieval
- Deterministic reranking
- Multi-cloud knowledge (AWS, Azure, GCP)

### Setup

In [1]:
import sys
from pathlib import Path
from dotenv import load_dotenv
import os
import chromadb
import re

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.settings import CHROMA_DIR, COLLECTION_NAME
from src.vectorstore import get_retriever
from src.rag_lcel import build_chain

load_dotenv(PROJECT_ROOT / "src" / ".env")
print("OPENAI_API_KEY loaded?:", bool(os.getenv("OPENAI_API_KEY")))

OPENAI_API_KEY loaded?: True


In [2]:
client = chromadb.PersistentClient(path=str(CHROMA_DIR))
col = client.get_collection(COLLECTION_NAME)

print("Count:", col.count())

INFO:chromadb.telemetry.product.posthog:Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.


Count: 74877


In [3]:
# Load chain
chain = build_chain(where=None)

single_cloud_q = "What is an Azure landing zone, and what governance problems does it solve?"
multi_cloud_q  = "Compare AWS Landing Zone, Azure Landing Zone, and GCP Landing Zone concepts. What governance problems do they solve and how do they differ?"

test_questions = {
    "single_cloud": single_cloud_q,
    "multi_cloud": multi_cloud_q,
}

results = {}

for test_name, question in test_questions.items():
    out = chain.invoke(question)

    # Step 3 scoring + grounding validation
    #grounding_check = validate_grounding(out)
    #score = score_answer(out)

    results[test_name] = {
        "question": question,
        "answer": out.get("answer", ""),
        "sources": out.get("sources", [])
    }


INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cpu
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
INFO:rag:[optionD:stage1_broad] Q='What is an Azure landing zone, and what governance problems does it solve?...' provider_counts={'azure': 8} total_docs=8
INFO:rag:[optionD:stage2_filtered(azure)] Q='What is an Azure landing zone, and what governance problems does it solve?...' provider_counts={'azure': 8} total_docs=8
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:rag:[topicAware:merged(topic=enterprise_cloud_foundation, k_per=5)] Q='Compare AWS Landing Zone, Azure Landing Zone, and GCP Landing Zone concepts. Wha...' provider_counts={'aws': 5, 'azure': 5, 'gcp': 5} total_docs=15
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


In [4]:
# Quick prints
print("\n==== SINGLE CLOUD ANSWER ====\n")
print(results["single_cloud"]["answer"])
print("\nSources:", results["single_cloud"]["sources"])

print("\n==== MULTI CLOUD ANSWER ====\n")
print(results["multi_cloud"]["answer"])
print("\nSources:", results["multi_cloud"]["sources"])


==== SINGLE CLOUD ANSWER ====

An **Azure landing zone** is a structured framework designed to help organizations set up and manage their Azure environments effectively. It provides a standardized approach to deploying cloud resources, ensuring that the architecture aligns with best practices for governance, security, and operational efficiency. Azure landing zones are divided into two main categories:

1. **Application Landing Zones**: These are specific Azure subscriptions where workloads run. They connect to shared platform resources, allowing access to essential infrastructure components such as networking, identity management, and monitoring.

2. **Platform Landing Zones**: These consist of multiple subscriptions managed by different platform teams, each serving a specific function (e.g., centralized DNS resolution or cross-premises connectivity) [2][4][8].

### Governance Problems Addressed by Azure Landing Zones

Azure landing zones tackle several governance challenges that org